In [1]:
from dofusapi import DofusAPI
from config import Config
from dataprocessor import DataProcessor
from utils import CacheManager

import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Set up plotting styles
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)


In [2]:
# Fetch equipment data
equipments = DofusAPI.get_all_equipments()

✅ 332 équipements récupérés avec succès


In [4]:
from collections import defaultdict

# Enhanced resource calculation with group filtering
def calculate_group_resources(groups, equipments, min_group_size=2):
    """
    Calculate the total resources needed for each equipment group.
    Only includes groups with at least min_group_size equipments.
    """
    # Filter groups by minimum size
    filtered_groups = [group for group in groups if len(group) >= min_group_size]
    
    # Create a mapping from equipment ID to equipment details
    equipment_dict = {equip['ankama_id']: equip for equip in equipments}
    
    group_resources = {}
    
    for group_idx, group in enumerate(filtered_groups):
        group_id = group_idx + 1
        total_resources = defaultdict(int)
        resource_details = defaultdict(list)
        
        # Calculate total resources needed
        for equipment in group:
            equip_id = equipment['ankama_id']
            for resource in equipment['recipe']:
                resource_id = resource['item_ankama_id']
                quantity = resource['quantity']
                total_resources[resource_id] += quantity
                resource_details[resource_id].append({
                    'equipment_id': equip_id,
                    'equipment_name': equipment['name'],
                    'quantity': quantity
                })
        
        # Calculate shared resources metrics
        shared_resources = defaultdict(int)
        for resource_id, usages in resource_details.items():
            if len(usages) > 1:  # Resource is shared by multiple equipments
                shared_resources[resource_id] = len(usages)
        
        group_resources[group_id] = {
            'equipment_count': len(group),
            'total_resources': dict(total_resources),
            'resource_details': dict(resource_details),
            'shared_resources_count': len(shared_resources),
            'shared_resources': dict(shared_resources),
            'total_quantity': sum(total_resources.values()),
            'unique_resources_count': len(total_resources),
            'equipment_list': [{'id': e['ankama_id'], 'name': e['name']} for e in group]
        }
    
    return group_resources

def print_group_resource_summary(group_resources):
    """
    Print a summary of resources needed for each group.
    """
    print("EQUIPMENT GROUP RESOURCE SUMMARY")
    print("=" * 50)
    
    for group_id, data in group_resources.items():
        print(f"\nGroup {group_id}:")
        print(f"  - Number of equipments: {data['equipment_count']}")
        print(f"  - Total resources needed: {data['total_quantity']}")
        print(f"  - Unique resources: {data['unique_resources_count']}")
        print(f"  - Shared resources: {data['shared_resources_count']}")
        
        # Print top 5 resources by quantity
        sorted_resources = sorted(data['total_resources'].items(), 
                                 key=lambda x: x[1], reverse=True)[:5]
        print("  - Top 5 resources by quantity:")
        for resource_id, quantity in sorted_resources:
            print(f"      Resource {resource_id}: {quantity} units")

def get_group_resource_details(group_resources, group_id):
    """
    Get detailed resource information for a specific group.
    """
    if group_id not in group_resources:
        return None
    
    data = group_resources[group_id]
    details = {
        'group_id': group_id,
        'equipment_count': data['equipment_count'],
        'total_resources': data['total_resources'],
        'shared_resources': data['shared_resources'],
        'total_quantity': data['total_quantity'],
        'unique_resources_count': data['unique_resources_count'],
        'shared_resources_count': data['shared_resources_count']
    }
    
    # Add equipment names
    equipment_names = set()
    for resource_usages in data['resource_details'].values():
        for usage in resource_usages:
            equipment_names.add(usage['equipment_name'])
    details['equipment_names'] = list(equipment_names)
    
    return details

In [ ]:
import os
import json
import time
import requests
from functools import lru_cache

# Initialize the cache manager at the start
CacheManager.initialize()

def print_group_resource_summary(group_resources):
    """
    Print a summary of resources needed for each group, using resource names.
    """
    print("EQUIPMENT GROUP RESOURCE SUMMARY")
    print("=" * 50)
    
    for group_id, data in group_resources.items():
        print(f"\nGroup {group_id}:")
        print(f"  - Number of equipments: {data['equipment_count']}")
        print(f"  - Total resources needed: {data['total_quantity']}")
        print(f"  - Unique resources: {data['unique_resources_count']}")
        print(f"  - Shared resources: {data['shared_resources_count']}")
        
        # Print top 5 resources by quantity with names
        sorted_resources = sorted(data['total_resources'].items(), 
                                 key=lambda x: x[1], reverse=True)[:5]
        print("  - Top 5 resources by quantity:")
        for resource_id, quantity in sorted_resources:
            resource_name = CacheManager.get_resource_name(resource_id)
            print(f"      {resource_name}: {quantity} units")

def get_group_resource_details(group_resources, group_id):
    """
    Get detailed resource information for a specific group, using resource names.
    """
    if group_id not in group_resources:
        return None
    
    data = group_resources[group_id]
    details = {
        'group_id': group_id,
        'equipment_count': data['equipment_count'],
        'total_resources': {},
        'shared_resources': {},
        'total_quantity': data['total_quantity'],
        'unique_resources_count': data['unique_resources_count'],
        'shared_resources_count': data['shared_resources_count']
    }
    
    # Add resource names to total_resources
    for resource_id, quantity in data['total_resources'].items():
        resource_name = CacheManager.get_resource_name(resource_id)
        details['total_resources'][resource_name] = quantity
    
    # Add resource names to shared_resources
    for resource_id, count in data['shared_resources'].items():
        resource_name = CacheManager.get_resource_name(resource_id)
        details['shared_resources'][resource_name] = count
    
    # Add equipment names
    equipment_names = set()
    for resource_usages in data['resource_details'].values():
        for usage in resource_usages:
            equipment_names.add(usage['equipment_name'])
    details['equipment_names'] = list(equipment_names)
    
    return details

def print_detailed_group_report(group_resources, group_id):
    """
    Print a detailed report for a specific group with resource names.
    """
    details = get_group_resource_details(group_resources, group_id)
    if not details:
        print(f"Group {group_id} not found.")
        return
    
    print(f"\nDETAILED REPORT FOR GROUP {group_id}")
    print("=" * 50)
    print(f"Equipment in this group: {', '.join(details['equipment_names'])}")
    print(f"Total resources needed: {details['total_quantity']}")
    print(f"Unique resources: {details['unique_resources_count']}")
    print(f"Shared resources: {details['shared_resources_count']}")
    
    print("\nResources needed:")
    for resource_name, quantity in details['total_resources'].items():
        print(f"  - {resource_name}: {quantity}")
    
    print("\nShared resources (used by multiple equipment):")
    for resource_name, count in details['shared_resources'].items():
        print(f"  - {resource_name}: used by {count} equipment")


In [6]:

def print_group_quality_report(metrics):
    """
    Print a comprehensive report on group quality.
    """
    print("EQUIPMENT GROUP QUALITY REPORT")
    print("=" * 50)
    print(f"Total groups: {metrics['total_groups']}")
    print(f"Total equipments in groups: {metrics['total_equipments']}")
    print(f"Average group size: {metrics['avg_group_size']:.2f}")
    print(f"Average shared resources per group: {metrics.get('avg_shared_resources', 0):.2f}")
    print(f"Average sharing efficiency: {metrics.get('avg_sharing_efficiency', 0):.2%}")
    print("\n")
    
    # Print details for each group
    for group in metrics['group_metrics']:
        print(f"Group {group['group_id']}:")
        print(f"  - Equipment count: {group['equipment_count']}")
        print(f"  - Equipment: {', '.join(group['equipment_names'])}")
        print(f"  - Total unique resources: {group['total_resources']}")
        print(f"  - Shared resources: {group['shared_resources']}")
        print(f"  - Sharing efficiency: {group['sharing_efficiency']:.2%}")
        if group['shared_resource_ids']:
            print(f"  - Shared resource IDs: {group['shared_resource_ids']}")
        print()

In [7]:
try:
    # Your equipment data here

    # First find the groups
    groups = DataProcessor().find_equipment_groups(equipments, resolution=20.0, min_group_size=3)
    
    # Calculate resources for each group
    group_resources = calculate_group_resources(groups, equipments)
    
    # Print summary with resource names
    print_group_resource_summary(group_resources)
    
    # Print detailed report for a specific group
    print_detailed_group_report(group_resources, 1)
    
finally:
    # Always save the cache when we're done
    CacheManager.save_cache()
    print(f"\nCache saved to {Config.CACHE_FILE}")

EQUIPMENT GROUP RESOURCE SUMMARY

Group 1:
  - Number of equipments: 3
  - Total resources needed: 53
  - Unique resources: 10
  - Shared resources: 3
  - Top 5 resources by quantity:
      Fleur de Kaliptus: 21 units
      Poils de Koalak Indigo: 9 units
      Poils de Koalak Griotte: 9 units
      Galet cramoisi: 4 units
      Tresse du Koulosse: 4 units

Group 2:
  - Number of equipments: 4
  - Total resources needed: 111
  - Unique resources: 13
  - Shared resources: 1
  - Top 5 resources by quantity:
      Rose des sables: 50 units
      Serrure du Coffre des Forgerons: 12 units
      Relique Familiale: 7 units
      Graine de la Discorde: 7 units
      Sueur Froide de Fantôme: 7 units

Group 3:
  - Number of equipments: 9
  - Total resources needed: 407
  - Unique resources: 36
  - Shared resources: 6
  - Top 5 resources by quantity:
      Pépite: 80 units
      Or: 80 units
      Roue de Tivelo: 26 units
      Racine d'Abraknyde Vénérable: 24 units
      Bourgeon de Pirolienne: 

In [8]:
def find_optimized_equipment_groups(equipments, min_shared_resources=2, 
                                   excluded_resource_ids=None, 
                                   resolution=1.0, min_group_size=2,
                                   max_group_size=8, efficiency_threshold=0.3):
    """
    Find equipment groups optimized for resource sharing and bulk efficiency,
    while excluding specific resources from the sharing calculation.
    
    Args:
        equipments: List of equipment dictionaries
        min_shared_resources: Minimum number of non-excluded shared resources required
        excluded_resource_ids: Set of resource IDs to exclude from sharing calculation
        resolution: Community detection resolution parameter
        min_group_size: Minimum equipment per group
        max_group_size: Maximum equipment per group
        efficiency_threshold: Minimum resource sharing efficiency required
    """
    if excluded_resource_ids is None:
        excluded_resource_ids = set()
    
    # Create the bipartite graph
    B = DataProcessor().create_bipartite_graph(equipments)
    
    # Get all equipment nodes
    equipment_nodes = {n for n, attr in B.nodes(data=True) if attr['bipartite'] == 0}
    
    # Project the bipartite graph to equipment graph with weights
    equipment_graph = bipartite.weighted_projected_graph(B, equipment_nodes)
    
    # Use Louvain community detection
    partition = community.best_partition(equipment_graph, resolution=resolution)
    
    # Group equipment nodes by community
    communities = {}
    for node, community_id in partition.items():
        communities.setdefault(community_id, []).append(node)
    
    # Map equipment ids back to equipment details
    equipment_dict = {equip['ankama_id']: equip for equip in equipments}
    groups = []
    
    for community_id, equip_ids in communities.items():
        group_equipments = []
        for equip_id in equip_ids:
            if isinstance(equip_id, str):
                try:
                    equip_id_int = int(equip_id)
                    if equip_id_int in equipment_dict:
                        group_equipments.append(equipment_dict[equip_id_int])
                except ValueError:
                    continue
            elif equip_id in equipment_dict:
                group_equipments.append(equipment_dict[equip_id])
        
        # Skip if group doesn't meet size requirements
        if not (min_group_size <= len(group_equipments) <= max_group_size):
            continue
        
        # Calculate shared resources (excluding specified resources)
        shared_count, total_shared, efficiency = calculate_shared_resources(
            group_equipments, excluded_resource_ids
        )
        
        # Apply multiple filters
        if (shared_count >= min_shared_resources and 
            efficiency >= efficiency_threshold):
            groups.append({
                'equipments': group_equipments,
                'shared_resources_count': shared_count,
                'total_shared_resources': total_shared,
                'sharing_efficiency': efficiency
            })
    
    # Sort groups by sharing efficiency (descending)
    groups.sort(key=lambda x: x['sharing_efficiency'], reverse=True)
    
    return groups

def calculate_shared_resources(group_equipments, excluded_resource_ids):
    """
    Calculate shared resources for a group, excluding specified resources.
    Returns:
        - Count of shared resources (excluding excluded ones)
        - Total shared resources (including excluded ones)
        - Sharing efficiency (shared_count / total_unique_resources)
    """
    resource_usage = defaultdict(int)
    all_resources = set()
    
    for equipment in group_equipments:
        for resource in equipment['recipe']:
            resource_id = resource['item_ankama_id']
            resource_usage[resource_id] += 1
            all_resources.add(resource_id)
    
    # Calculate shared resources (excluding specified ones)
    shared_resources = {
        rid: count for rid, count in resource_usage.items() 
        if count > 1 and rid not in excluded_resource_ids
    }
    
    # Calculate total shared resources (including excluded ones)
    total_shared_resources = {
        rid: count for rid, count in resource_usage.items() 
        if count > 1
    }
    
    # Calculate sharing efficiency
    total_unique = len(all_resources)
    shared_count = len(shared_resources)
    efficiency = shared_count / total_unique if total_unique > 0 else 0
    
    return shared_count, len(total_shared_resources), efficiency

def print_optimized_groups(groups, excluded_resource_ids=None):
    """
    Print optimized groups with resource sharing information.
    """
    if excluded_resource_ids is None:
        excluded_resource_ids = set()
    
    print("OPTIMIZED EQUIPMENT GROUPS")
    print("=" * 50)
    print(f"Excluded resources from sharing calculation: {excluded_resource_ids}")
    print(f"Total groups found: {len(groups)}")
    print("\n")
    
    for i, group in enumerate(groups):
        print(f"Group {i+1}:")
        print(f"  - Equipment count: {len(group['equipments'])}")
        print(f"  - Shared resources (non-excluded): {group['shared_resources_count']}")
        print(f"  - Total shared resources: {group['total_shared_resources']}")
        print(f"  - Sharing efficiency: {group['sharing_efficiency']:.2%}")
        
        # List equipment in this group
        print("  - Equipment:")
        for equip in group['equipments']:
            print(f"      {equip['name']} (Level: {equip['level']})")
        
        # List shared resources (excluding specified ones)
        shared_resources = get_shared_resources(group['equipments'], excluded_resource_ids)
        if shared_resources:
            print("  - Shared resources (non-excluded):")
            for resource_id, count in shared_resources.items():
                resource_name = CacheManager.get_resource_name(resource_id)
                print(f"      {resource_name} (used by {count} equipment)")
        
        print()

def get_shared_resources(group_equipments, excluded_resource_ids):
    """
    Get shared resources for a group, excluding specified resources.
    """
    resource_usage = defaultdict(int)
    
    for equipment in group_equipments:
        for resource in equipment['recipe']:
            resource_id = resource['item_ankama_id']
            resource_usage[resource_id] += 1
    
    # Return only shared resources that aren't excluded
    return {
        rid: count for rid, count in resource_usage.items() 
        if count > 1 and rid not in excluded_resource_ids
    }


In [9]:
def find_optimized_equipment_groups(equipments, min_shared_resources=2, 
                                   excluded_resource_ids=None, 
                                   resolution=1.0, min_group_size=2,
                                   max_group_size=8, efficiency_threshold=0.3):
    """
    Find equipment groups optimized for resource sharing and bulk efficiency,
    while excluding specific resources from the sharing calculation.
    
    Args:
        equipments: List of equipment dictionaries
        min_shared_resources: Minimum number of non-excluded shared resources required
        excluded_resource_ids: Set of resource IDs to exclude from sharing calculation
        resolution: Community detection resolution parameter
        min_group_size: Minimum equipment per group
        max_group_size: Maximum equipment per group
        efficiency_threshold: Minimum resource sharing efficiency required
    """
    if excluded_resource_ids is None:
        excluded_resource_ids = set()
    
    # Create the bipartite graph
    B = DataProcessor().create_bipartite_graph(equipments)
    
    # Get all equipment nodes
    equipment_nodes = {n for n, attr in B.nodes(data=True) if attr['bipartite'] == 0}
    
    # Project the bipartite graph to equipment graph with weights
    equipment_graph = bipartite.weighted_projected_graph(B, equipment_nodes)
    
    # Use Louvain community detection
    partition = community.best_partition(equipment_graph, resolution=resolution)
    
    # Group equipment nodes by community
    communities = {}
    for node, community_id in partition.items():
        communities.setdefault(community_id, []).append(node)
    
    # Map equipment ids back to equipment details
    equipment_dict = {equip['ankama_id']: equip for equip in equipments}
    groups = []
    
    for community_id, equip_ids in communities.items():
        group_equipments = []
        for equip_id in equip_ids:
            if isinstance(equip_id, str):
                try:
                    equip_id_int = int(equip_id)
                    if equip_id_int in equipment_dict:
                        group_equipments.append(equipment_dict[equip_id_int])
                except ValueError:
                    continue
            elif equip_id in equipment_dict:
                group_equipments.append(equipment_dict[equip_id])
        
        # Skip if group doesn't meet size requirements
        if not (min_group_size <= len(group_equipments) <= max_group_size):
            continue
        
        # Calculate shared resources (excluding specified resources)
        shared_count, total_shared, efficiency = calculate_shared_resources(
            group_equipments, excluded_resource_ids
        )
        
        # Calculate total ingredients needed for the group
        total_ingredients = calculate_total_ingredients(group_equipments)
        
        # Apply multiple filters
        if (shared_count >= min_shared_resources and 
            efficiency >= efficiency_threshold):
            groups.append({
                'equipments': group_equipments,
                'shared_resources_count': shared_count,
                'total_shared_resources': total_shared,
                'sharing_efficiency': efficiency,
                'total_ingredients': total_ingredients,
                'unique_ingredients_count': len(total_ingredients),
                'total_items_needed': sum(ingredient['total_quantity'] for ingredient in total_ingredients.values())
            })
    
    # Sort groups by sharing efficiency (descending)
    groups.sort(key=lambda x: x['sharing_efficiency'], reverse=True)
    
    return groups

def calculate_total_ingredients(group_equipments):
    """
    Calculate the total ingredients needed for all equipment in the group.
    Returns a dictionary with resource_id as key and aggregated ingredient info as value.
    """
    ingredients = defaultdict(lambda: {
        'name': None,
        'total_quantity': 0,
        'used_in_equipments': [],
        'quantity_per_equipment': {}
    })
    
    for equipment in group_equipments:
        for resource in equipment['recipe']:
            resource_id = resource['item_ankama_id']
            quantity = resource['quantity']
            
            # Update ingredient information
            ingredients[resource_id]['total_quantity'] += quantity
            ingredients[resource_id]['used_in_equipments'].append(equipment['name'])
            ingredients[resource_id]['quantity_per_equipment'][equipment['name']] = quantity
            
            # Get resource name if not already set
            if ingredients[resource_id]['name'] is None:
                ingredients[resource_id]['name'] = CacheManager.get_resource_name(resource_id)
    
    return dict(ingredients)

def calculate_shared_resources(group_equipments, excluded_resource_ids):
    """
    Calculate shared resources for a group, excluding specified resources.
    Returns:
        - Count of shared resources (excluding excluded ones)
        - Total shared resources (including excluded ones)
        - Sharing efficiency (shared_count / total_unique_resources)
    """
    resource_usage = defaultdict(int)
    all_resources = set()
    
    for equipment in group_equipments:
        for resource in equipment['recipe']:
            resource_id = resource['item_ankama_id']
            resource_usage[resource_id] += 1
            all_resources.add(resource_id)
    
    # Calculate shared resources (excluding specified ones)
    shared_resources = {
        rid: count for rid, count in resource_usage.items() 
        if count > 1 and rid not in excluded_resource_ids
    }
    
    # Calculate total shared resources (including excluded ones)
    total_shared_resources = {
        rid: count for rid, count in resource_usage.items() 
        if count > 1
    }
    
    # Calculate sharing efficiency
    total_unique = len(all_resources)
    shared_count = len(shared_resources)
    efficiency = shared_count / total_unique if total_unique > 0 else 0
    
    return shared_count, len(total_shared_resources), efficiency

def print_optimized_groups(groups, excluded_resource_ids=None, show_ingredients=True):
    """
    Print optimized groups with resource sharing information and full ingredient list.
    """
    if excluded_resource_ids is None:
        excluded_resource_ids = set()
    
    print("OPTIMIZED EQUIPMENT GROUPS")
    print("=" * 80)
    print(f"Excluded resources from sharing calculation: {excluded_resource_ids}")
    print(f"Total groups found: {len(groups)}")
    print("\n")
    
    for i, group in enumerate(groups):
        print(f"Group {i+1}:")
        print(f"  - Equipment count: {len(group['equipments'])}")
        print(f"  - Shared resources (non-excluded): {group['shared_resources_count']}")
        print(f"  - Total shared resources: {group['total_shared_resources']}")
        print(f"  - Sharing efficiency: {group['sharing_efficiency']:.2%}")
        print(f"  - Unique ingredients needed: {group['unique_ingredients_count']}")
        print(f"  - Total items needed: {group['total_items_needed']}")
        
        # List equipment in this group
        print("  - Equipment:")
        for equip in group['equipments']:
            print(f"      {equip['name']} (Level: {equip['level']})")
        
        # List shared resources (excluding specified ones)
        shared_resources = get_shared_resources(group['equipments'], excluded_resource_ids)
        if shared_resources:
            print("  - Shared resources (non-excluded):")
            for resource_id, count in shared_resources.items():
                resource_name = CacheManager.get_resource_name(resource_id)
                print(f"      {resource_name} (used by {count} equipment)")
        
        # Print full ingredient list if requested
        if show_ingredients and group['total_ingredients']:
            print("  - Full ingredient list:")
            # Sort ingredients by total quantity (descending)
            sorted_ingredients = sorted(
                group['total_ingredients'].items(), 
                key=lambda x: x[1]['total_quantity'], 
                reverse=True
            )
            
            for resource_id, ingredient_info in sorted_ingredients:
                usage_details = []
                for equip_name, quantity in ingredient_info['quantity_per_equipment'].items():
                    usage_details.append(f"{equip_name}({quantity})")
                
                usage_str = ", ".join(usage_details)
                print(f"      {ingredient_info['name']}: {ingredient_info['total_quantity']} total")
                print(f"          Used in: {usage_str}")
        
        print()

def get_shared_resources(group_equipments, excluded_resource_ids):
    """
    Get shared resources for a group, excluding specified resources.
    """
    resource_usage = defaultdict(int)
    
    for equipment in group_equipments:
        for resource in equipment['recipe']:
            resource_id = resource['item_ankama_id']
            resource_usage[resource_id] += 1
    
    # Return only shared resources that aren't excluded
    return {
        rid: count for rid, count in resource_usage.items() 
        if count > 1 and rid not in excluded_resource_ids
    }

def export_ingredients_to_csv(groups, filename="equipment_group_ingredients.csv"):
    """
    Export group ingredients to a CSV file for further analysis.
    """
    import csv
    
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            'Group', 'Equipment_Count', 'Sharing_Efficiency', 
            'Resource_ID', 'Resource_Name', 'Total_Quantity',
            'Used_In_Equipments', 'Quantity_Per_Equipment'
        ])
        
        for i, group in enumerate(groups):
            for resource_id, ingredient_info in group['total_ingredients'].items():
                # Format equipment usage as string
                usage_pairs = [
                    f"{equip}:{qty}" for equip, qty 
                    in ingredient_info['quantity_per_equipment'].items()
                ]
                usage_str = "; ".join(usage_pairs)
                
                writer.writerow([
                    i+1,
                    len(group['equipments']),
                    f"{group['sharing_efficiency']:.2%}",
                    resource_id,
                    ingredient_info['name'],
                    ingredient_info['total_quantity'],
                    ", ".join(ingredient_info['used_in_equipments']),
                    usage_str
                ])
    
    print(f"Ingredients exported to {filename}")

# Additional utility function to get ingredient summary
def get_ingredient_summary(groups):
    """
    Get a summary of all ingredients across all groups.
    """
    all_ingredients = defaultdict(int)
    
    for group in groups:
        for resource_id, ingredient_info in group['total_ingredients'].items():
            all_ingredients[resource_id] += ingredient_info['total_quantity']
    
    # Convert to sorted list
    summary = []
    for resource_id, total_quantity in all_ingredients.items():
        resource_name = CacheManager.get_resource_name(resource_id)
        summary.append({
            'resource_id': resource_id,
            'resource_name': resource_name,
            'total_quantity': total_quantity
        })
    
    summary.sort(key=lambda x: x['total_quantity'], reverse=True)
    return summary

In [10]:
import json
import os
from pathlib import Path

def generate_group_visualization(group, group_index, output_dir="visualizations"):
    """
    Generate an interactive HTML visualization for a single equipment group.
    """
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Prepare the graph data
    graph_data = prepare_graph_data(group)
    
    # Generate the HTML content
    html_content = create_visualization_html(graph_data, group, group_index)
    
    # Write to file
    filename = f"{output_dir}/group_{group_index}_visualization.html"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return filename

def prepare_graph_data(group):
    """
    Prepare the graph data structure for D3.js visualization.
    """
    nodes = []
    links = []
    
    # Add equipment nodes
    equipment_nodes = {}
    for i, equipment in enumerate(group['equipments']):
        node_id = f"equip_{equipment['ankama_id']}"
        equipment_nodes[equipment['ankama_id']] = node_id
        nodes.append({
            'id': node_id,
            'name': equipment['name'],
            'type': 'equipment',
            'level': equipment.get('level', 'N/A')
        })
    
    # Add resource nodes and links
    resource_nodes = {}
    for resource_id, ingredient_info in group['total_ingredients'].items():
        node_id = f"res_{resource_id}"
        resource_nodes[resource_id] = node_id
        nodes.append({
            'id': node_id,
            'name': ingredient_info['name'],
            'type': 'resource',
            'total_quantity': ingredient_info['total_quantity']
        })
        
        # Create links from resources to equipment
        for equip_name, quantity in ingredient_info['quantity_per_equipment'].items():
            # Find the equipment node ID
            equip_id = None
            for equipment in group['equipments']:
                if equipment['name'] == equip_name:
                    equip_id = equipment_nodes[equipment['ankama_id']]
                    break
            
            if equip_id:
                links.append({
                    'source': node_id,
                    'target': equip_id,
                    'quantity': quantity
                })
    
    return {'nodes': nodes, 'links': links}

def generate_all_visualizations(groups, output_dir="visualizations"):
    """
    Generate visualizations for all groups and create an index page.
    """
    # Create individual group visualizations
    visualization_files = []
    for i, group in enumerate(groups):
        filename = generate_group_visualization(group, i+1, output_dir)
        visualization_files.append({
            'group_index': i+1,
            'filename': filename,
            'group': group
        })
    
    # Create index page
    create_index_page(visualization_files, output_dir)
    
    return visualization_files

def create_index_page(visualization_files, output_dir):
    """
    Create an index page with links to all group visualizations.
    """
    index_content = """
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>Equipment Groups - Visualizations</title>
            <style>
                body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 40px; background: #f5f5f5; }
                .container { max-width: 800px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
                h1 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
                .group-list { list-style: none; padding: 0; }
                .group-item { background: #f8f9fa; margin: 10px 0; padding: 15px; border-radius: 5px; border-left: 4px solid #3498db; }
                .group-link { text-decoration: none; color: #2c3e50; font-weight: bold; font-size: 1.1em; }
                .group-stats { font-size: 0.9em; color: #6c757d; margin-top: 5px; }
            </style>
        </head>
        <body>
            <div class="container">
                <h1>Equipment Group Visualizations</h1>
                <ul class="group-list">
        """
    
    for viz in visualization_files:
        group = viz['group']
        index_content += f"""
            <li class="group-item">
                <a href="{os.path.basename(viz['filename'])}" class="group-link">
                    Group {viz['group_index']} - {len(group['equipments'])} Equipment Items
                </a>
                <div class="group-stats">
                    {group['unique_ingredients_count']} unique resources • 
                    {group['total_items_needed']} total items • 
                    {group['sharing_efficiency']:.1%} sharing efficiency
                </div>
            </li>
"""
    
    index_content += """
        </ul>
    </div>
</body>
</html>
"""
    
    with open(f"{output_dir}/index.html", 'w', encoding='utf-8') as f:
        f.write(index_content)

# Usage example:
def visualize_equipment_groups(groups):
    """
    Main function to generate all visualizations.
    """
    print("Generating equipment group visualizations...")
    
    # Generate all visualizations
    visualization_files = generate_all_visualizations(groups)
    
    print(f"Generated {len(visualization_files)} visualization files:")
    for viz in visualization_files:
        print(f"  - {viz['filename']}")
    print(f"Index page: visualizations/index.html")
    print("\nOpen 'visualizations/index.html' in your browser to view all groups.")
    
    return visualization_files

# Integrate with your existing code:
def enhanced_print_optimized_groups(groups, excluded_resource_ids=None, generate_visualizations=True):
    """
    Enhanced version that can generate visualizations.
    """
    if excluded_resource_ids is None:
        excluded_resource_ids = set()
    
    print("OPTIMIZED EQUIPMENT GROUPS")
    print("=" * 80)
    print(f"Excluded resources from sharing calculation: {excluded_resource_ids}")
    print(f"Total groups found: {len(groups)}")
    print("\n")
    
    for i, group in enumerate(groups):
        print(f"Group {i+1}:")
        print(f"  - Equipment count: {len(group['equipments'])}")
        print(f"  - Shared resources (non-excluded): {group['shared_resources_count']}")
        print(f"  - Total shared resources: {group['total_shared_resources']}")
        print(f"  - Sharing efficiency: {group['sharing_efficiency']:.2%}")
        print(f"  - Unique ingredients: {group['unique_ingredients_count']}")
        print(f"  - Total items needed: {group['total_items_needed']}")
        
        # List equipment in this group
        print("  - Equipment:")
        for equip in group['equipments']:
            print(f"      {equip['name']} (Level: {equip['level']})")
        
        # List shared resources (excluding specified ones)
        shared_resources = get_shared_resources(group['equipments'], excluded_resource_ids)
        if shared_resources:
            print("  - Shared resources (non-excluded):")
            for resource_id, count in shared_resources.items():
                resource_name = CacheManager.get_resource_name(resource_id)
                print(f"      {resource_name} (used by {count} equipment)")
        
        print()
    
    # Generate visualizations if requested
    if generate_visualizations and groups:
        visualize_equipment_groups(groups)
    
    return groups

In [11]:


# Define resources to exclude from sharing calculation
# These might be rare, expensive, or otherwise problematic resources
excluded_resources = {15263, 14635}  # Example resource IDs

# Find optimized groups
groups = find_optimized_equipment_groups(
    equipments,
    min_shared_resources=3,
    excluded_resource_ids=excluded_resources,
    resolution=4,
    min_group_size=3,
    max_group_size=6,
    efficiency_threshold=0.25
)
len(groups)

11

In [12]:

from visualizer import EquipmentVisualizer

viz = EquipmentVisualizer()
resource_info_getter = CacheManager().get_resource_info

# viz.create_simple_test_page()
viz.generate_visualizations(groups, resource_info_getter)
# visualize_equipment_groups(groups, CacheManager().get_resource_info)


🎯 Generating professional visualizations for 11 groups...
✅ Created group_1.html
✅ Created group_2.html
✅ Created group_3.html
✅ Created group_4.html
✅ Created group_5.html
✅ Created group_6.html
✅ Created group_7.html
✅ Created group_8.html
✅ Created group_9.html
✅ Created group_10.html
✅ Created group_11.html
✅ Created index.html

🎉 Professional visualizations generated!
👉 Open http://localhost:8000/index.html in your browser
📊 Features included:
   • Interactive force-directed graphs
   • Complete ingredient tables with quantities
   • Professional modern design
   • Drag-and-drop interactivity


In [13]:
groups = DataProcessor().find_equipment_groups(equipments, resolution=1.0, min_group_size=2)
    
# Calculate resources for each group
group_resources = calculate_group_resources(groups, equipments)

# Print summary with resource names
print_group_resource_summary(group_resources)

# Print detailed report for a specific group
print_detailed_group_report(group_resources, 1)

# You can also create a function to print all detailed reports
def print_all_detailed_reports(group_resources):
    for group_id in group_resources.keys():
        print_detailed_group_report(group_resources, group_id)
        print("\n" + "="*50 + "\n")

EQUIPMENT GROUP RESOURCE SUMMARY

Group 1:
  - Number of equipments: 87
  - Total resources needed: 1308
  - Unique resources: 171
  - Shared resources: 65
  - Top 5 resources by quantity:
      Substrat de Futaie: 102 units
      Peau de Shin Larve: 36 units
      Serrure du Coffre des Forgerons: 35 units
      Cuir de Porkass: 35 units
      Relique Familiale: 34 units

Group 2:
  - Number of equipments: 59
  - Total resources needed: 3424
  - Unique resources: 105
  - Shared resources: 57
  - Top 5 resources by quantity:
      Rose des sables: 2050 units
      Pépite: 100 units
      Bakélélite: 92 units
      Étoffe de Dok Alako: 84 units
      Carapace du Mantiscore: 64 units

Group 3:
  - Number of equipments: 49
  - Total resources needed: 748
  - Unique resources: 115
  - Shared resources: 35
  - Top 5 resources by quantity:
      Magnésite: 59 units
      Peau de Ramane d'Égoutant: 32 units
      Sacrum magistral: 26 units
      Queue de Rat d'Hyoactif: 25 units
      Bourse S

In [14]:
from dataprocessor import DataProcessor
groups = DataProcessor().find_equipment_groups(equipments, resolution=10.0, min_group_size=2)

# Evaluate group quality
metrics = DataProcessor().evaluate_group_quality(groups, equipments)

# Print the report
print_group_quality_report(metrics)

# You can also filter groups by sharing efficiency
efficient_groups = [
    group for group, metrics in zip(groups, metrics['group_metrics']) 
    if metrics['sharing_efficiency'] > 0.3  # At least 30% of resources are shared
]

print(f"Found {len(efficient_groups)} groups with high resource sharing efficiency")

EQUIPMENT GROUP QUALITY REPORT
Total groups: 42
Total equipments in groups: 209
Average group size: 4.98
Average shared resources per group: 4.29
Average sharing efficiency: 32.29%


Group 1:
  - Equipment count: 7
  - Equipment: Pelle Woukuis, Blopture Multicolore Royale, Bloptes Multicolores Royales, Arc du Pêcheur, Dagues Ruyère, God Rod, La Griffe Aiguisée
  - Total unique resources: 20
  - Shared resources: 8
  - Sharing efficiency: 40.00%
  - Shared resource IDs: [2508, 9386, 748, 2496, 9384, 9385, 9387, 9389]

Group 2:
  - Equipment count: 2
  - Equipment: Bâton du Koulosse, Arc du Koalak
  - Total unique resources: 8
  - Shared resources: 2
  - Sharing efficiency: 25.00%
  - Shared resource IDs: [7904, 8062]

Group 3:
  - Equipment count: 2
  - Equipment: Bottes Ouroboulos, Bardiche du Milicien
  - Total unique resources: 8
  - Shared resources: 3
  - Sharing efficiency: 37.50%
  - Shared resource IDs: [18366, 18385, 2543]

Group 4:
  - Equipment count: 9
  - Equipment: Bottes 